# CityPilot IA · Cuaderno de validación

**Objetivo:** usar el contenedor Jupyter como **cliente independiente** de la web y PostgreSQL. Completa las celdas, ejecútalas y guarda los resultados en el repositorio. No es un notebook decorativo: los 4 controles son obligatorios.

> Dentro del contenedor, la URL de Flask es `http://web:5000`; `localhost:5000` apuntaría al propio Jupyter. Los datos de los fixtures son de demostración; no confundirlos con respuestas vivas.

In [ ]:
import os
import requests
import pandas as pd
from sqlalchemy import create_engine, text

WEB = os.environ.get("WEB_BASE_URL", "http://web:5000")
DB = os.environ["DATABASE_URL"]
print("API:", WEB)
print("Base configurada:", bool(DB))

## 0. Salud de la arquitectura (pista resuelta)
Comprueba que Jupyter puede llegar a Flask y que Flask responde sobre la BD. Explica por escrito qué significa cada conexión.

In [ ]:
respuesta = requests.get(f"{WEB}/api/health", timeout=5)
print(respuesta.status_code, respuesta.json())
assert respuesta.status_code == 200

## 1. API de datos públicos (obligatorio)
Con `requests`, recupera **Valencia → coordenadas → clima actual**. No uses lat/lon de memoria como respuesta final: debes sacarlas del JSON de Geocoding. Muestra ciudad, lat, lon, temperatura y precipitación. Comprueba qué haces si `results` está vacío.

En `docs/APIS_Y_CONTRATOS.md` tienes URLs y parámetros. Puedes poner los campos `name`, `count`, `countryCode`, `current`, `timezone` en diccionarios de parámetros.

In [ ]:
geo_url = "https://geocoding-api.open-meteo.com/v1/search"
meteo_url = "https://api.open-meteo.com/v1/forecast"

# 1) Geocoding: de "Valencia" a coordenadas (sacadas del JSON, no de memoria)
geo = requests.get(
    geo_url,
    params={"name": "Valencia", "count": 1, "language": "es", "countryCode": "ES"},
    timeout=10,
)
geo.raise_for_status()
resultados = geo.json().get("results")
if not resultados:
    # Caso 'results' vacío: la ciudad no existe o no se ha encontrado
    raise ValueError("Open-Meteo no ha devuelto resultados para esa ciudad")
lugar = resultados[0]
ciudad, lat, lon = lugar["name"], lugar["latitude"], lugar["longitude"]

# 2) Meteorología actual con esas coordenadas
meteo = requests.get(
    meteo_url,
    params={
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,precipitation,weather_code",
        "timezone": "auto",
    },
    timeout=10,
)
meteo.raise_for_status()
actual = meteo.json()["current"]

# 3) Evidencia
print("Ciudad:", ciudad)
print("Latitud:", lat, "| Longitud:", lon)
print("Temperatura (°C):", actual["temperature_2m"])
print("Precipitación (mm):", actual["precipitation"])


## 2. API de vuestro Flask (obligatorio)
Desde **Jupyter** haz una petición POST JSON a `/api/planificar`. Comprueba: status 200, campos `itinerario`, `meteorologia`, `lugares`, `solicitud_id` y que la ciudad coincide con la enviada. Usa un tipo del conjunto aceptado. Si no tenéis clave Cohere, `LLM_MODE=mock` permite validar transporte y flujo; **no demuestra generación real**.

In [ ]:
payload = {
    "ciudad": "Valencia",
    "tipo_turismo": "cultural",
    "duracion_horas": 4,
    "preferencias": "poco caminar",
}
respuesta = requests.post(f"{WEB}/api/planificar", json=payload, timeout=120)
print("Status:", respuesta.status_code)
datos = respuesta.json()

assert respuesta.status_code == 200, datos
for campo in ("itinerario", "meteorologia", "lugares", "solicitud_id"):
    assert campo in datos, f"Falta el campo {campo}"
assert datos["ciudad"].lower() == payload["ciudad"].lower()

print("solicitud_id:", datos["solicitud_id"])
print("Ciudad:", datos["ciudad"], "| modo_demo:", datos["modo_demo"])
print("Lugares recibidos:", len(datos["lugares"]))
print("--- Itinerario ---")
print(datos["itinerario"])

## 3. PostgreSQL (obligatorio)
Después de crear rutas mediante API, conecta a PostgreSQL desde este notebook. Obtén el número de solicitudes y una tabla `id, fecha, ciudad, tipo_turismo, estado` con hasta cinco filas. Compara con el número y últimas filas que ves en vuestro dashboard.

In [ ]:
engine = create_engine(DB, pool_pre_ping=True)
with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM solicitudes")).scalar_one()
print("Filas en PostgreSQL:", n)

ultimas = pd.read_sql_query(
    text("""
        SELECT id, fecha, ciudad, tipo_turismo, estado
        FROM solicitudes
        ORDER BY fecha DESC, id DESC
        LIMIT 5
    """),
    engine,
)
ultimas

## 4. Error controlado (obligatorio)
Elige un caso de validación: ciudad vacía, `duracion_horas=99` o `tipo_turismo=espacial`. Haz `POST /api/planificar` y demuestra que la aplicación responde **400** con un mensaje claro. Una respuesta 500 inesperada no supera esta prueba.

In [ ]:
# Entrada inválida: tipo de turismo que no existe
invalido = {
    "ciudad": "Valencia",
    "tipo_turismo": "espacial",
    "duracion_horas": 4,
    "preferencias": "",
}
respuesta = requests.post(f"{WEB}/api/planificar", json=invalido, timeout=30)
print("Status:", respuesta.status_code)
print("Mensaje:", respuesta.json())
assert respuesta.status_code == 400
assert "error" in respuesta.json()

## 5. Bonus · Analítica desde el notebook
Con `pandas.read_sql_query` y SQLAlchemy, compara las ciudades y tipos más solicitados. Si te interesa, haz aquí una figura de Matplotlib; la figura obligatoria del dashboard **debe seguir saliendo del servidor web y de PostgreSQL**, no copiarse manualmente desde el notebook.

In [ ]:
# BONUS: ciudades y tipos más solicitados, leídos directamente de PostgreSQL
por_ciudad = pd.read_sql_query(
    text("SELECT ciudad, COUNT(*) AS solicitudes FROM solicitudes GROUP BY ciudad ORDER BY solicitudes DESC"),
    engine,
)
por_tipo = pd.read_sql_query(
    text("SELECT tipo_turismo, COUNT(*) AS solicitudes FROM solicitudes GROUP BY tipo_turismo ORDER BY solicitudes DESC"),
    engine,
)
display(por_ciudad)
display(por_tipo)
por_ciudad.plot(kind="bar", x="ciudad", y="solicitudes", legend=False, title="Solicitudes por ciudad");

### Evidencia final
Guarda el notebook con salidas visibles, sin claves y sin capturas sensibles. En vuestro README, describe brevemente **una conclusión** de los datos almacenados. ¡Cuidado! Las filas `demo=true` son simuladas y se deben identificar como tales.

**Semáforo amarillo:** en tu memoria individual indica si la IA te ayudó a escribir celdas, consultas o asserts; explica qué corregiste y dónde se ve tu prueba. No incluir claves.